In [1]:
import os
import chromadb
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter   # ← langchain.text_splitter 에서 변경
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# ── 엔드포인트 ─────────────────────────────────────────────
LMSTUDIO_BASE_URL = "http://...:.../v1"  # /v1 까지 붙여야 함
LMSTUDIO_API_KEY  = "lm-studio"                              # LM Studio는 키 검증 안 함 → 더미 값

EMBED_MODEL = "text-embedding-qwen3-embedding-8b"
LLM_MODEL   = "qwen3-30b-a3b-instruct-2507"

CHROMA_HOST = "chromadb"
CHROMA_PORT = 8000
COLLECTION  = "kb2024"

PDF_PATH = "test.pdf"  # 2024 KB 부동산 보고서
print("PDF_PATH =", PDF_PATH, "| 존재:", os.path.exists(PDF_PATH))

/tmp/ipykernel_25803/3648918791.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader
Skipping import of cpp extensions due to incompatible torch version 2.10.0+cu128 for torchao version 0.14.0         Please see GitHub issue #2919 for more info


PDF_PATH = test.pdf | 존재: True


In [2]:
embedding_function = OpenAIEmbeddings(
    model=EMBED_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key=LMSTUDIO_API_KEY,
    check_embedding_ctx_length=False,   # ★ 토큰ID 대신 원문 문자열 전송
)

# 연결 확인: 차원 수가 찍히면 OK
_v = embedding_function.embed_query("연결 테스트")
print("임베딩 차원:", len(_v))

임베딩 차원: 4096


In [3]:
chroma_client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
print("heartbeat:", chroma_client.heartbeat())  # ns 값이 나오면 연결 OK

heartbeat: 1782301463668837576


In [4]:
vectorstore = Chroma(
    client=chroma_client,
    collection_name=COLLECTION,
    embedding_function=embedding_function,
)

count = vectorstore._collection.count()
if count == 0:
    print("컬렉션이 비어 있음 → 인덱싱 시작")
    documents = PyPDFLoader(PDF_PATH).load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = splitter.split_documents(documents)
    print("분할된 청크 수:", len(chunks))
    vectorstore.add_documents(chunks)   # LM Studio 임베딩으로 벡터화 후 원격 적재
    print("적재 완료. 문서 수:", vectorstore._collection.count())
else:
    print("이미 적재됨. 문서 수:", count, "→ 인덱싱 건너뜀")

컬렉션이 비어 있음 → 인덱싱 시작
분할된 청크 수: 135
적재 완료. 문서 수: 135


In [5]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

template = """당신은 KB 부동산 보고서 전문가입니다. 다음 정보를 바탕으로 사용자의 질문에 답변해주세요.

컨텍스트: {context}
"""
prompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("placeholder", "{chat_history}"),
    ("human", "{question}"),
])

model = ChatOpenAI(
    model=LLM_MODEL,
    base_url=LMSTUDIO_BASE_URL,
    api_key="lm_studio",   # 임의 문자열
    temperature=0.7,
)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

base_chain = (
    RunnablePassthrough.assign(
        context=lambda x: format_docs(retriever.invoke(x["question"]))
    )
    | prompt
    | model
    | StrOutputParser()
)

store = {}
def get_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

chain = RunnableWithMessageHistory(
    base_chain,
    get_history,
    input_messages_key="question",
    history_messages_key="chat_history",
)

/opt/venv/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [6]:
# 빠른 검증 (서비스 가동 전 노트북에서 직접 확인)
resp = chain.invoke(
    {"question": "수도권 주택 매매 전망을 알려줘"},
    {"configurable": {"session_id": "nb-test"}},
)
print(resp)

다음은 **2024 KB 부동산 보고서**에 기반한 **수도권 주택 매매 전망**에 대한 종합적 요약입니다:

---

### ✅ **1. 전체 시장 동향: 하락세 후 회복 기대, 양극화 심화**
- 2023년 수도권 주택 매매가격은 **연간 4.64% 하락**하며, 전년 대비 하락폭이 **2배 이상 확대**됨.
- 거래량은 월 평균 **5만 호 내외로 침체 상태 지속**, 수요 회복이 원활하지 않음.
- 하지만 **2023년 상반기 이후 회복세 진입**, 특히 12월 하락 전환 후 다시 반등 기대 형성.

> 🔍 **핵심 요인**: 높은 금리, DSR 규제 등으로 매수력 제한 지속 → 그러나 정부의 **규제 완화**로 매도자들의 기대 심리 상승.

---

### ✅ **2. 주요 지역별 전망**

#### 📍 **서울 한강이남 (강남구)**
- **매매가격 하락폭 최소**, 수요 집중으로 안정성 유지.
- 재건축 규제 완화 + 신속통합기획 추진 → 압구정동 아파트는 **30억 → 44억 원** 상승 (최고가 기록).
- 전세시장도 학군수요(신학기 이사)로 인해 **상승세 지속**.
- 시장 양극화 심화: 강남 등 선호 지역으로 수요 집중 → 매물 감소, 호가 유지.

> ✅ **전망**: 정부 규제 완화 기대 + 수요 집중 덕분에 **강남권은 안정적 상승세 예상**.  
> ⚠️ 다만 거래 활성화는 지연될 가능성이 높음.

#### 📍 **서울 한강이북 (마포구, 성동구 등)**
- 마포구: 지역 간 매매가격 격차 확대 → 수요 변화 발생.
- 성동구: 대기 수요 많지만 실제 가입 여부에 따라 전망 결정적.

> ✅ **전망**: 호재(한남뉴타운 등) 있음 → 긍정적 전환 가능성 있으나, 매도자와 매수자 간 **이격** 지속.

#### 📍 **위례**
- 위례선 개통 예정 → 긍정적 요인.
- 하지만 **위례신사선 착공 지연**으로 부정적 요인이 혼재됨.
> ✅ 전망: 개발 속도에 따라 양면적 전망. 단기적으로는 불확실성 존재.

#### 📍 **과천**

In [7]:
%%writefile app.py
import os
import chromadb
import streamlit as st
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

LMSTUDIO_BASE_URL = os.getenv("LMSTUDIO_BASE_URL", "http://...:.../v1")
EMBED_MODEL = os.getenv("EMBED_MODEL", "text-embedding-qwen3-embedding-8b")
LLM_MODEL   = os.getenv("LLM_MODEL", "qwen3-30b-a3b-instruct-2507")
CHROMA_HOST = os.getenv("CHROMA_HOST", "chromadb")
CHROMA_PORT = int(os.getenv("CHROMA_PORT", "8000"))
COLLECTION  = os.getenv("COLLECTION", "kb2024")
PDF_PATH    = os.getenv("PDF_PATH", "./test.pdf")


@st.cache_resource
def get_embeddings():
    return OpenAIEmbeddings(
        model=EMBED_MODEL,
        base_url=LMSTUDIO_BASE_URL,
        api_key="lm-studio",
        check_embedding_ctx_length=False,   # ★ 토큰ID 대신 원문 문자열 전송
    )


@st.cache_resource
def initialize_vectorstore():
    client = chromadb.HttpClient(host=CHROMA_HOST, port=CHROMA_PORT)
    vs = Chroma(client=client, collection_name=COLLECTION, embedding_function=get_embeddings())
    if vs._collection.count() == 0:
        documents = PyPDFLoader(PDF_PATH).load()
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
        vs.add_documents(splitter.split_documents(documents))
    return vs


@st.cache_resource
def initialize_chain():
    vectorstore = initialize_vectorstore()
    retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

    template = """당신은 KB 부동산 보고서 전문가입니다. 다음 정보를 바탕으로 사용자의 질문에 답변해주세요.

컨텍스트: {context}
"""
    prompt = ChatPromptTemplate.from_messages([
        ("system", template),
        ("placeholder", "{chat_history}"),
        ("human", "{question}"),
    ])
    model = ChatOpenAI(
        model=LLM_MODEL,
        base_url=LMSTUDIO_BASE_URL,
        api_key="lmstudio",
        temperature=0.7,
    )

    def format_docs(docs):
        return "\n\n".join(doc.page_content for doc in docs)

    base_chain = (
        RunnablePassthrough.assign(
            context=lambda x: format_docs(retriever.invoke(x["question"]))
        )
        | prompt
        | model
        | StrOutputParser()
    )

    store = {}

    def get_history(session_id: str):
        if session_id not in store:
            store[session_id] = ChatMessageHistory()
        return store[session_id]

    return RunnableWithMessageHistory(
        base_chain,
        get_history,
        input_messages_key="question",
        history_messages_key="chat_history",
    )


def main():
    st.set_page_config(page_title="KB 부동산 보고서 챗봇", page_icon="\U0001F3E0")
    st.title("\U0001F3E0 KB 부동산 보고서 AI 어드바이저")
    st.caption("2024 KB 부동산 보고서 기반 질의응답 시스템")

    if "messages" not in st.session_state:
        st.session_state.messages = []

    for message in st.session_state.messages:
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

    if prompt := st.chat_input("부동산 관련 질문을 입력하세요"):
        with st.chat_message("user"):
            st.markdown(prompt)
        st.session_state.messages.append({"role": "user", "content": prompt})

        chain = initialize_chain()
        with st.chat_message("assistant"):
            with st.spinner("답변 생성 중..."):
                response = chain.invoke(
                    {"question": prompt},
                    {"configurable": {"session_id": "streamlit_session"}},
                )
            st.markdown(response)
        st.session_state.messages.append({"role": "assistant", "content": response})


if __name__ == "__main__":
    main()

Writing app.py


In [8]:
!pip install -q streamlit

In [9]:
import subprocess, sys

proc = subprocess.Popen([
    sys.executable, "-m", "streamlit", "run", "app.py",
    "--server.port", "8501",
    "--server.address", "0.0.0.0",
    "--server.headless", "true",
    "--server.enableCORS", "false",
    "--server.enableXsrfProtection", "false",
])
print("Streamlit started on container port 8501")
# 중지하려면: proc.terminate()

In [10]:
proc.terminate()

  Stopping...


In [11]:
import subprocess, time, socket
subprocess.run(["pkill", "-f", "streamlit run"])
time.sleep(2)
s = socket.socket(); r = s.connect_ex(("127.0.0.1", 8501)); s.close()
print("8501:", "아직 점유중" if r == 0 else "비었음(정리됨)")

8501: 비었음(정리됨)
